<a href="https://colab.research.google.com/github/eeeaaai/YL/blob/main/FedAsync_deneme_versiyon_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import threading
import time
import random
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# ----------------------------
# Global Settings and Variables
# ----------------------------
NUM_CLIENTS = 5
MAX_GLOBAL_ITERATIONS = 50 #Async Federated Optimization parameter T
ALPHA = 0.5            # Base mixing weight
BATCH_SIZE = 64

global_lock = threading.Lock()
global_iteration = 0   # Tracks the number of global updates

# ----------------------------
# Define the CNN for MNIST Classification
# ----------------------------
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.dropout = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)  # 10 output classes for MNIST

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.dropout(self.conv2(x)), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Global model shared among clients
global_model = CNN()

# ----------------------------
# Utility Functions
# ----------------------------
def staleness_decay(tau):
    """Simple decay function: as tau (staleness) increases, the update weight decreases."""
    return 1.0 / (tau + 1)

def update_global_model(client_state, client_timestamp):
    """Update the global model with a client's state using a staleness-adjusted mixing weight."""
    global global_model, global_iteration
    with global_lock:
        tau = global_iteration - client_timestamp
        decay = staleness_decay(tau)
        adjusted_alpha = ALPHA * decay

        # Get the current global state and mix in the client's parameters
        global_state = global_model.state_dict()
        for key in global_state:
            global_state[key] = (1 - adjusted_alpha) * global_state[key] + adjusted_alpha * client_state[key]
        global_model.load_state_dict(global_state)

        global_iteration += 1
        print(f"[Server] Global iter: {global_iteration}, tau: {tau}, adjusted_alpha: {adjusted_alpha:.4f}")

# ----------------------------
# Client Training Function
# ----------------------------
def client_training(client_id, train_loader, num_local_epochs=1):
    """
    Each client:
      - Copies the current global model.
      - Trains locally on its MNIST partition for a number of epochs.
      - Sends its updated parameters to the server.
    """
    global global_model, global_iteration
    criterion = nn.CrossEntropyLoss()
    updates_done = 0
    local_loss_history = []  # Track training losses for this client

    while True:
        with global_lock:
            if global_iteration >= MAX_GLOBAL_ITERATIONS:
                break
            # Copy the global model
            local_model = CNN()
            local_model.load_state_dict(global_model.state_dict())
            current_timestamp = global_iteration

        optimizer = optim.SGD(local_model.parameters(), lr=0.01)
        local_model.train()

        # Train for the specified number of local epochs
        for epoch in range(num_local_epochs):
            for batch_idx, (data, target) in enumerate(train_loader):
                optimizer.zero_grad()
                output = local_model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()
                local_loss_history.append(loss.item())

                if batch_idx % 100 == 0:
                    print(f"[Client {client_id}] Epoch {epoch} Batch {batch_idx} Loss: {loss.item():.4f}")

        print(f"[Client {client_id}] Finished local update #{updates_done}, Last Loss: {loss.item():.4f}")
        update_global_model(local_model.state_dict(), current_timestamp)
        updates_done += 1

        time.sleep(random.uniform(0.5, 2.0))  # Simulate asynchronous delay

    print(f"[Client {client_id}] Completed training after {updates_done} updates. Loss history (first 5): {local_loss_history[:5]}")
    return local_loss_history

# ----------------------------
# Global Evaluation Function
# ----------------------------
def evaluate_global_model(test_loader):
    """Evaluate the global model on the MNIST test set."""
    global global_model
    global_model.eval()
    criterion = nn.CrossEntropyLoss(reduction='sum')
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            output = global_model(data)
            test_loss += criterion(output, target).item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()

    test_loss /= len(test_loader.dataset)
    accuracy = correct / len(test_loader.dataset)
    print(f"\n[Evaluation] Average loss: {test_loss:.4f}, Accuracy: {accuracy*100:.2f}%\n")
    return test_loss, accuracy

# ----------------------------
# Main Function
# ----------------------------
def main():
    # Define transforms and load MNIST dataset
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,))
    ])
    train_dataset = datasets.MNIST('./data', train=True, download=True, transform=transform)
    test_dataset  = datasets.MNIST('./data', train=False, download=True, transform=transform)

    # Split the training dataset into NUM_CLIENTS parts
    train_size = len(train_dataset)
    split_size = train_size // NUM_CLIENTS
    client_datasets = random_split(train_dataset, [split_size] * NUM_CLIENTS)
    client_loaders = [DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True) for ds in client_datasets]

    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    # Start asynchronous client threads
    clients = []
    for client_id in range(NUM_CLIENTS):
        t = threading.Thread(target=client_training, args=(client_id, client_loaders[client_id]))
        t.start()
        clients.append(t)

    # Wait for all clients to finish training
    for t in clients:
        t.join()

    # Evaluate the final global model on the test set
    evaluate_global_model(test_loader)

if __name__ == "__main__":
    main()


Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [00:00<00:00, 17.2MB/s]


Extracting ./data/MNIST/raw/train-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 491kB/s]


Extracting ./data/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1.65M/1.65M [00:00<00:00, 3.85MB/s]


Extracting ./data/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4.54k/4.54k [00:00<00:00, 2.27MB/s]


Extracting ./data/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/MNIST/raw

[Client 2] Epoch 0 Batch 0 Loss: 2.3070
[Client 0] Epoch 0 Batch 0 Loss: 2.3165
[Client 1] Epoch 0 Batch 0 Loss: 2.3180
[Client 3] Epoch 0 Batch 0 Loss: 2.3342
[Client 4] Epoch 0 Batch 0 Loss: 2.3050
[Client 1] Epoch 0 Batch 100 Loss: 2.2818
[Client 2] Epoch 0 Batch 100 Loss: 2.2717
[Client 3] Epoch 0 Batch 100 Loss: 2.2422
[Client 0] Epoch 0 Batch 100 Loss: 2.2621[Client 4] Epoch 0 Batch 100 Loss: 2.2719

[Client 0] Finished local update #0, Last Loss: 2.1275[Client 4] Finished local update #0, Last Loss: 2.0802

[Server] Global iter: 1, tau: 0, adjusted_alpha: 0.5000
[Server] Global iter: 2, tau: 1, adjusted_alpha: 0.2500
[Client 1] Finished local update #0, Last Loss: 2.0950
[Server] Global iter: 3, tau: 2, adjusted_alpha: 0.1667
[Client 3] Finished local update #0, Last Loss: 2.0462
[Server] Global iter: 4, tau: 3, adjusted_alpha: 0.1250
[Client 2] Finished local update #0, Last Loss: 2.0655
[Server] Global 